# Chương 2 — Học máy cơ bản: MLP cài đặt bằng ba cách trên dữ liệu tiểu đường

In [1]:
# ML-00 · Thiết lập đường dẫn: mã chương dùng chung gói của tiểu luận (src/dm) và mã kế thừa của nhóm (src/tl01, tl03)
import os, sys, json, time, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
from pathlib import Path
warnings.filterwarnings("ignore")
TLF = Path.cwd().parent
for p in ["src", "src/tl01", "src/tl03"]:
    sys.path.insert(0, str(TLF / p))
import numpy as np, pandas as pd
np.set_printoptions(precision=4, suppress=True)
print("Thư mục tiểu luận:", TLF)

Thư mục tiểu luận: /home/ngtukien/Projects/MultiAgent/tieuluan_final


Notebook chạy lại phần cốt lõi của Chương 2 trên bộ **Diabetes Prediction (100.000 hồ sơ)**: tiền xử lý không rò rỉ, cùng một MLP `Dense(128)–ReLU–Dropout–Dense(64)–ReLU–Dropout–Dense(1)` viết bằng NumPy, Keras và PyTorch, rồi đọc lại kết quả đầy đủ do `scripts/run_ch2_*.py` ghi ra.

In [2]:
# ML-01 · Nạp dữ liệu và xem phân bố nhãn
from tl import data_tabular as dt
df = dt.diabetes_frame()
print(df.shape)
print(df["diabetes"].value_counts(normalize=True).round(4))
df.head()

(100000, 9)
diabetes
0    0.915
1    0.085
Name: proportion, dtype: float64


,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


In [3]:
# ML-02 · Chia 80/20 phân tầng TRƯỚC, rồi mới khớp bộ chuẩn hóa trên tập huấn luyện (không rò rỉ)
d = dt.load_diabetes()
Xtr, Xte, ytr, yte = d["X_train"], d["X_test"], d["y_train"], d["y_test"]
print("train", Xtr.shape, "test", Xte.shape, "tỷ lệ dương", ytr.mean().round(4), yte.mean().round(4))
num = d["pre"].named_transformers_["num"]
print("trung bình dùng để chuẩn hóa (chỉ tính trên train):", np.round(num.mean_, 3))

train (80000, 15) test (20000, 15) tỷ lệ dương 0.085 0.085
trung bình dùng để chuẩn hóa (chỉ tính trên train): [ 41.901  27.317   5.528 138.079]


In [4]:
# ML-03 · MLP viết tay bằng NumPy: lan truyền tiến, ngược và Adam — huấn luyện thử 5 epoch
from tl.mlp_scratch import MLPScratch
from sklearn.metrics import roc_auc_score
m = MLPScratch(Xtr.shape[1], output_dim=1, task="binary")
t0 = time.time()
m.fit(Xtr[:60000], ytr[:60000], Xtr[60000:], ytr[60000:], epochs=5, log=lambda *a: None)
p = m.predict_raw(Xte).ravel()
print(f"NumPy: AUC = {roc_auc_score(yte, p):.4f} sau 5 epoch, {time.time()-t0:.1f}s")

NumPy: AUC = 0.9737 sau 5 epoch, 2.7s


In [5]:
# ML-04 · Cùng kiến trúc bằng Keras và PyTorch (5 epoch) để so cách viết
import os; os.environ["TL_TF_THREADS"] = "4"
from tl.mlp_keras import build_keras_mlp, fit as kfit
from tl.mlp_torch import TorchMLP, fit as tfit, predict_raw
km = build_keras_mlp(Xtr.shape[1], task="binary")
kfit(km, Xtr[:60000], ytr[:60000], Xtr[60000:], ytr[60000:], epochs=5)
print("Keras  : AUC =", round(roc_auc_score(yte, km.predict(Xte, verbose=0).ravel()), 4))
tm = TorchMLP(Xtr.shape[1])
tfit(tm, Xtr[:60000], ytr[:60000], Xtr[60000:], ytr[60000:], "binary", epochs=5, log=lambda *a: None)
print("PyTorch: AUC =", round(roc_auc_score(yte, predict_raw(tm, Xte).ravel()), 4))
print(km.summary())

I0000 00:00:1793839297.229237  223839 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


I0000 00:00:1793839298.837653  223839 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Keras  : AUC = 0.9747


PyTorch: AUC = 0.9754


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         2,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,109 (121.52 KB)

 Trainable params: 10,369 (40.50 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 20,740 (81.02 KB)

None


In [6]:
# ML-05 · Đọc kết quả đầy đủ (40 epoch, dừng sớm) của ba framework và năm thuật toán cổ điển
M = TLF / "src/tl01/outputs/metrics"
rows = []
for fw in ["scratch", "keras", "torch"]:
    r = json.loads((M / f"ch2_mlp_{fw}.json").read_text())["diabetes"]
    rows.append({"mô hình": f"MLP – {fw}", **{k: r[k] for k in ["acc", "f1", "auc", "train_s", "infer_ms"]}})
sk = json.loads((M / "ch2_sklearn.json").read_text())          # khoá dạng "diabetes/logreg"
rows += [{"mô hình": k.split("/")[1], **{c: v[c] for c in ["acc", "f1", "auc", "train_s", "infer_ms"]}}
         for k, v in sk.items() if k.startswith("diabetes/")]
pd.DataFrame(rows).round(4)

,mô hình,acc,f1,auc,train_s,infer_ms
0,MLP – scratch,0.9721,0.8043,0.9775,32.2358,0.0030
1,MLP – keras,0.9723,0.8063,0.9771,61.4235,0.0275
2,MLP – torch,0.9722,0.8059,0.9777,20.1345,0.0004
3,logreg,0.8888,0.5776,0.9629,1.0293,0.0035
4,rf,0.9639,0.7755,0.9738,9.7584,0.0301
5,svm,0.8958,0.5997,0.9718,240.3200,1.2442


In [7]:
# ML-06 · Ngưỡng quyết định: Youden J so với 0,5 cố định (minh họa trên tập kiểm tra; thực tế chọn trên validation)
from sklearn.metrics import roc_curve
fpr, tpr, thr = roc_curve(yte, p)
j = np.argmax(tpr - fpr)
print(f"ngưỡng Youden = {thr[j]:.3f}: độ nhạy {tpr[j]:.3f}, độ đặc hiệu {1-fpr[j]:.3f}")
for t in [0.3, 0.5, 0.7]:
    pred = p >= t
    print(f"ngưỡng {t}: độ nhạy {pred[yte==1].mean():.3f}, độ đặc hiệu {(~pred)[yte==0].mean():.3f}")

ngưỡng Youden = 0.101: độ nhạy 0.922, độ đặc hiệu 0.895
ngưỡng 0.3: độ nhạy 0.735, độ đặc hiệu 0.982
ngưỡng 0.5: độ nhạy 0.655, độ đặc hiệu 0.997
ngưỡng 0.7: độ nhạy 0.608, độ đặc hiệu 1.000
